In [2]:
import os
import cv2
import numpy as np
import random
from tqdm import tqdm

def vit_specific_augmentation(img):
    """
    Augmentations tailored for ViT:
    Focuses on Random Resized Crops and Color Jitter.
    """
    h, w = img.shape[:2]

    # 1. Random Resized Crop (Crucial for ViT Patch learning)
    # Scales between 70% and 100% of the image
    scale = random.uniform(0.7, 1.0)
    new_h, new_w = int(h * scale), int(w * scale)
    top = random.randint(0, h - new_h)
    left = random.randint(0, w - new_w)
    img = img[top:top+new_h, left:left+new_w]
    img = cv2.resize(img, (224, 224)) # Resize back to ViT standard input

    # 2. Horizontal Flip
    if random.random() > 0.5:
        img = cv2.flip(img, 1)

    # 3. Color Jitter (Brightness and Contrast)
    brightness = random.uniform(0.8, 1.2)
    contrast = random.uniform(0.8, 1.2)
    img = cv2.convertScaleAbs(img, alpha=contrast, beta=int(10 * brightness))

    # 4. Gaussian Blur (To force ViT to look at global structures, not just noise)
    if random.random() > 0.8:
        img = cv2.GaussianBlur(img, (3, 3), 0)

    return img

def create_vit_augmented_dataset(source_root, target_root):
    # Matches your 4-class pneumonia structure
    classes = ['Covid-19', 'Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral']

    for cls in classes:
        source_dir = os.path.join(source_root, cls)
        target_dir = os.path.join(target_root, cls)
        os.makedirs(target_dir, exist_ok=True)

        if not os.path.exists(source_dir):
            print(f"Skipping {cls}: Source directory not found.")
            continue

        images = [f for f in os.listdir(source_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        print(f"\n✨ Augmenting {cls} for ViT...")

        for img_name in tqdm(images):
            img_path = os.path.join(source_dir, img_name)
            img = cv2.imread(img_path)
            if img is None: continue

            # Standard ViT Input Resize
            img = cv2.resize(img, (224, 224))

            # Save the original (Base for Equal Sample)
            cv2.imwrite(os.path.join(target_dir, f"base_{img_name}"), img)

            # Generate 1 Augmented Version
            # (You can loop this if you need more than 2x data)
            aug_img = vit_specific_augmentation(img)
            cv2.imwrite(os.path.join(target_dir, f"vit_aug_{img_name}"), aug_img)

    print(f"\n✅ ViT Augmented Data saved to: {target_root}")

# --- PATHS (Based on your Project Tree) ---
SOURCE = "/Users/kpvarma/PycharmProjects/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification/Equal_sample"
DESTINATION = "/Users/kpvarma/PycharmProjects/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification/Vit_Augumented_Data"

create_vit_augmented_dataset(SOURCE, DESTINATION)


✨ Augmenting Covid-19 for ViT...


100%|██████████| 2696/2696 [00:02<00:00, 1080.28it/s]



✨ Augmenting Normal for ViT...


100%|██████████| 2696/2696 [00:02<00:00, 1041.83it/s]



✨ Augmenting Pneumonia-Bacterial for ViT...


100%|██████████| 2696/2696 [00:02<00:00, 1091.08it/s]



✨ Augmenting Pneumonia-Viral for ViT...


100%|██████████| 2696/2696 [00:02<00:00, 1107.45it/s]


✅ ViT Augmented Data saved to: /Users/kpvarma/PycharmProjects/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification/Vit_Augumented_Data
